In [1]:
1*63

63

In [8]:
import os
import io
import datetime
from PIL import Image
import base64
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
from langchain_pinecone import PineconeVectorStore
from groq import Groq

In [3]:
pdf_path=Path('NovaCore_Multimodal_Company_Report_2026.pdf')
if not pdf_path.exists():
    fallback=Path("data/NovaCore_Multimodal_Company_Report_2026.pdf")
    if fallback.exists():
        pdf_path=fallback
assert pdf_path.exists(),(
    "pdf not found please keep pdf in same folder as notebook"
)
    

In [4]:
pdf_path


WindowsPath('NovaCore_Multimodal_Company_Report_2026.pdf')

In [7]:
TEXT_MODEL='google/gemini-3.6-flash'
VISION_MODEL = "qwen/qwen3.8-27b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PINECONE_INDEX_NAME = "novacore-multimodal-rag"
PINECONE_NAMESPACE = "fy2026-demo"

IMAGE_DIR = Path("novacore_extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

print("PDF:", pdf_path)
print("Pinecone index:", PINECONE_INDEX_NAME)
print("Pinecone namespace:", PINECONE_NAMESPACE)

PDF: NovaCore_Multimodal_Company_Report_2026.pdf
Pinecone index: novacore-multimodal-rag
Pinecone namespace: fy2026-demo


In [11]:
from groq import Groq
from langchain_groq import ChatGroq
if not os.getenv('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY']=''
if not os.getenv('PINECONE_API_KEY'):
    os.environ['PINECONE_API_KEY']=''
groq_client=Groq(api_key=os.getenv('GROQ_API_KEY'))

text_llm=ChatGroq(
    model=TEXT_MODEL,
    temperature=0
)

In [14]:
def image_to_data_uri(image_path):
    image_path=Path(image_path)
    with Image.open(image_path) as image:
        image=image.convert('RGB')
        image.thumbnail((1600,1600))
        buffer=io.BytesIO()
        image.save(buffer,format='jpeg',quality=85)
    encoded=base64.b64encode(buffer.getvalue().decode('utf-8'))
    return f"data:image/jpeg;base64,{encoded}"

In [17]:
def summarize_visual(image_path, page_number):
    image_data = image_to_data_uri(image_path)

    prompt = f"""This visual was extracted from page {page_number} of the NovaCore FY2026 company report.

Describe the useful business information visible in the visual.

If it is a chart or graph:
- mention important values
- mention highest/lowest values
- mention the main trend

If it is a diagram:
- identify important components
- explain the flow or relationships

If it is a normal business image:
- describe the useful factual information

Keep the description concise and factual.
"""

    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_data},
                    },
                ],
            }
        ],
        temperature=0,
        max_completion_tokens=2000,
    )

    return response.choices[0].message.content.strip()